# Pipeline de Ingestão da Tabela de Votação

Notebook responsável pela ingestão da tabela **votacao** utilizando PySpark e Delta Lake. O fluxo realiza a leitura dos arquivos CSV brutos, consolida os dados em um DataFrame e grava a camada Bronze no formato Delta.

#### 1. Importação das Bibliotecas

Nesta etapa importamos as bibliotecas necessárias para realizar a ingestão da tabela **consulta_candidatos**. Utilizamos o Spark para processamento distribuído e o Delta Lake para persistência dos dados na camada Bronze.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


#### 2. Inicialização da Sessão Spark

Inicializamos a sessão Spark configurada para trabalhar com o Delta Lake. Essa sessão será utilizada durante todo o processo de ingestão da tabela de votação.

In [2]:
builder = (
    SparkSession.builder
        # .master('local[*]')
        .appName('TSE-Analytics-Validation')
        .config(
            'spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension'
        )
        .config(
            'spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog'
        )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print('Sessão Spark criada com sucesso com suporte a Delta Lake!')
print(f'Versão do PySpark: {spark.version}')

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-666b0640-9dc6-47d7-8f7b-b4366e0cf3b2;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.0 in central
	found io.delta#delta-storage;4.3.0 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.0 in

Sessão Spark criada com sucesso com suporte a Delta Lake!
Versão do PySpark: 4.1.1


#### 3. Leitura dos Arquivos da Tabela de Candidatos e Carga na Camada Raw (Parquet)

Lemos os arquivos CSV brutos de votações nas eleições de 2000 a 2024. Aplicamos uma padronização rigorosa nos tipos das colunas identificadoras (como `SQ_CANDIDATO`e `SG_UE`) para garantir consistência física entre diferentes partições anuais no formato **Parquet**. Os arquivos limpos são salvos por ano na Camada Raw.

In [3]:
anos = list(range(2000, 2025, 2))

for ano in anos:

    df = (spark.read
               .format('csv')
               .option('header', 'true')    
               .option('sep', ';')
               .option('encoding', 'ISO-8859-1')
               .option('InferSchema', 'true')
               .load(f'data/{ano}/votacao_candidato_munzona_*/*_BRASIL.csv'))
    
     # Padronização de tipos de colunas para evitar conflitos de tipos no Parquet e erros de conversão de esquema
    if "SQ_CANDIDATO" in df.columns:
        df = df.withColumn("SQ_CANDIDATO", F.col("SQ_CANDIDATO").cast("bigint"))

    if "SQ_COLIGACAO" in df.columns:
        df = df.withColumn("SQ_COLIGACAO", F.col("SQ_COLIGACAO").cast("bigint"))

    if "SG_UE" in df.columns:
        df = df.withColumn("SG_UE", F.col("SG_UE").cast("string"))

    (df.coalesce(1).write
                   .format('parquet')
                   .mode('overwrite')
                   .save(f'data/raw/votacao/{ano}'))

26/07/04 13:20:12 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/2000/votacao_candidato_munzona_*/*_BRASIL.csv.
java.io.FileNotFoundException: File data/2000/votacao_candidato_munzona_*/*_BRASIL.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource

#### 4. Consolidação e Carga na Camada Bronze (Delta lake)

Após a leitura dos arquivos CSV de todos os anos, realizamos a leitura consolidada dos arquivos Parquet gerados e gravamos a tabela **votacao** no formato Delta. Essa etapa representa a ingestão da tabela na camada Bronze do Data Lake.

In [4]:
df = (spark.read
           .format('parquet')
           .option('inferSchema', 'true')
           .load('data/raw/votacao/*'))

(df.coalesce(1)
   .write
   .format('delta')
   .mode('overwrite')
   .save('data/bronze/votacao'))

26/07/04 13:42:41 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/votacao/*.
java.io.FileNotFoundException: File data/raw/votacao/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa